# Preprocess

Explanation

Table of contents

## Utils


def 

In [ ]:
import pandas as pd
import ast

df = pd.read_csv("/Users/yavuzlule/Desktop/bsc-relish/data/external/recipe1m/full_dataset.csv")

In [189]:
df = df[:100]

df.head()

,Unnamed: 0,title,ingredients,directions,link,source,NER
0,0,No-Bake Nut Cookies,1 c. firmly packed brown sugar 1/2 c. evaporat...,"[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu..."
1,1,Jewell Ball'S Chicken,"1 small jar chipped beef, cut up 4 boned chick...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""beef"", ""chicken breasts"", ""cream of mushroom..."
2,2,Creamy Corn,2 (16 oz.) pkg. frozen corn 1 (8 oz.) pkg. cre...,"[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""cream cheese"", ""butter"", ""gar..."
3,3,Chicken Funny,1 large whole chicken 2 (10 1/2 oz.) cans chic...,"[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken"", ""chicken gravy"", ""cream of mushroo..."
4,4,Reeses Cups(Candy),1 c. peanut butter 3/4 c. graham cracker crumb...,"[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""peanut butter"", ""graham cracker crumbs"", ""bu..."


In [190]:
def parse_list_column(x):
    """
    Converts stringified list → list of strings.
    Handles:
    - valid Python list strings
    - already-list inputs
    - NaN / malformed cases
    """
    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
            return [str(parsed)]
        except Exception:
            # fallback: treat as single text blob
            return [x]

    return []

In [191]:
def join_list_text(lst):
    """
    Joins list of strings into a single text field.
    """
    lst = [str(i).strip() for i in lst if str(i).strip()]
    return " ".join(lst)

In [192]:
df["ingredients"] = df["ingredients"].apply(parse_list_column).apply(join_list_text)
df["directions"]  = df["directions"].apply(parse_list_column).apply(join_list_text)


df.head()

,Unnamed: 0,title,ingredients,directions,link,source,NER
0,0,No-Bake Nut Cookies,1 c. firmly packed brown sugar 1/2 c. evaporat...,"In a heavy 2-quart saucepan, mix brown sugar, ...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu..."
1,1,Jewell Ball'S Chicken,"1 small jar chipped beef, cut up 4 boned chick...",Place chipped beef on bottom of baking dish. P...,www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""beef"", ""chicken breasts"", ""cream of mushroom..."
2,2,Creamy Corn,2 (16 oz.) pkg. frozen corn 1 (8 oz.) pkg. cre...,"In a slow cooker, combine all ingredients. Cov...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""cream cheese"", ""butter"", ""gar..."
3,3,Chicken Funny,1 large whole chicken 2 (10 1/2 oz.) cans chic...,Boil and debone chicken. Put bite size pieces ...,www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken"", ""chicken gravy"", ""cream of mushroo..."
4,4,Reeses Cups(Candy),1 c. peanut butter 3/4 c. graham cracker crumb...,Combine first four ingredients and press in 13...,www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""peanut butter"", ""graham cracker crumbs"", ""bu..."


In [194]:
df = df[["title", "ingredients", "directions"]].copy()

In [195]:
df["text"] = df["title"].astype(str) + " " + df["ingredients"].astype(str) + " " + df["directions"].astype(str)
df["title"] = df["title"].astype(str)

df["title_start"] = 0
df["title_end"] = df["title"].str.len()

In [201]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")


In [198]:
def align_labels(text, start, end, tokenizer, max_length=512):
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

    offsets = encoding["offset_mapping"]
    labels = []

    for (s, e) in offsets:
        # special tokens
        if s == 0 and e == 0:
            labels.append(-100)
            continue

        # token overlaps title span
        if s >= start and e <= end:
            labels.append(1)
        else:
            labels.append(0)

    encoding["labels"] = labels
    return encoding

In [199]:
processed = [
    align_labels(
        row["text"],
        row["title_start"],
        row["title_end"],
        tokenizer
    )
    for _, row in df.iterrows()
]


from datasets import Dataset

dataset = Dataset.from_list(processed)

In [202]:
from transformers import AutoModelForTokenClassification

id2label = {0: "O", 1: "TITLE"}
label2id = {"O": 0, "TITLE": 1}

model = AutoModelForTokenClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 18958.72it/s]
[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [206]:
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]



In [207]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [208]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

def compute_metrics(p):
    logits, labels = p
    preds = np.argmax(logits, axis=-1)

    true_preds = []
    true_labels = []

    for pred, label in zip(preds, labels):
        for p_i, l_i in zip(pred, label):
            if l_i != -100:
                true_preds.append(p_i)
                true_labels.append(l_i)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, true_preds, average="binary"
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [223]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="./xlmr-title-ner",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=20,
    weight_decay=0.01,
    eval_strategy="epoch",   # <-- NEW (replaces evaluation_strategy)
    #save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

ValueError: --load_best_model_at_end requires the save and eval strategy to match, except when --save_strategy="best", but found
- Evaluation strategy: IntervalStrategy.EPOCH
- Save strategy: SaveStrategy.STEPS

In [224]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.000237,1.000000,1.000000,1.000000
2,No log,0.000093,1.000000,1.000000,1.000000
3,No log,0.000042,1.000000,1.000000,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.04it/s]
/Users/yavuzlule/Desktop/bsc-relish/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.26s/it]
/Users/yavuzlule/Desktop/bsc-relish/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.76s/it]


RuntimeError: [enforce fail at inline_container.cc:672] . unexpected pos 58944 vs 58836

In [216]:
text = "Chocolate Cake In a bowl mix flour sugar eggs. Bake for 30 minutes."
title = "Chocolate Cake"

start = 0
end = len(title)


encoding = tokenizer(
    text,
    return_offsets_mapping=True,
    return_tensors="pt",
    truncation=True,
    padding=True
)

offsets = encoding["offset_mapping"][0]
input_ids = encoding["input_ids"]
attention_mask = encoding["attention_mask"]


labels = []

for s, e in offsets:
    if s == 0 and e == 0:
        labels.append(-100)
    elif s >= start and e <= end:
        labels.append(1)
    else:
        labels.append(0)

labels = torch.tensor([labels])

In [219]:
import torch

device = torch.device("cpu")
model = model.to(device)

with torch.no_grad():

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

logits = outputs.logits
preds = torch.argmax(logits, dim=-1)


tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

for t, p in zip(tokens, preds[0]):
    print(f"{t:15} {p.item()}")



<s>             0
▁Chocolate      0
▁Ca             0
ke              0
▁In             0
▁a              0
▁bowl           0
▁mix            0
▁flo            0
ur              0
▁sugar          0
▁egg            0
s               0
.               0
▁Bak            0
e               0
▁for            0
▁30             0
▁minutes        0
.               0
</s>            0
